# GDV – Torangriffe, schnelle Transition und die 4–5-Pass-Beobachtung bei der FIFA WM 2022

## Leitfrage

Entstehen Tore bei der FIFA WM 2022 eher durch schnelle Angriffe mit wenigen Pässen oder durch längeren Spielaufbau?

## Praxisbezug

Die Idee kommt direkt aus dem Fussballtraining. Im Trainerumfeld wird häufig gesagt, dass viele Tore nach ungefähr vier bis fünf Pässen entstehen. Das ist keine feste Beobachtung, sondern eine verbreitete Beobachtung aus der Praxis. In diesem Notebook wird diese Aussage mit Eventdaten der FIFA WM 2022 überprüft.

Der praktische Nutzen ist klar: Ein Trainer oder Spieler möchte wissen, ob eine Mannschaft im Turnierfussball eher schnell vertikal spielen sollte oder ob längerer Aufbau vor Toren eine ähnlich grosse Rolle spielt. Für einen Verein kann daraus abgeleitet werden, ob im Training mehr Fokus auf schnelles Umschalten, Entries ins offensive Drittel oder geduldigen Spielaufbau gelegt werden sollte.

## Kernfragen

1. Wie viele erfolgreiche Pässe gehen einem Tor voraus?
2. Wie häufig liegen Tore im Bereich von vier bis fünf Pässen?
3. Kommen Teams schnell ins offensive Drittel, und entstehen daraus Abschlüsse?
4. Erfolgen Entries ins offensive Drittel eher über lange Bälle, kurze Kombinationen oder Carries?
5. Wo starten Torangriffe, die später zu Toren führen?
6. Welche Teams erzielen ihre Tore besonders direkt?
7. Wie sieht ein konkretes Tor im 4–5-Pass-Bereich aus?

## Wichtige Einschränkung

Die Analyse nutzt Eventdaten und keine Trackingdaten. Sie zeigt Ballaktionen wie Pässe, Carries und Schüsse. Sie zeigt aber nicht die Laufwege aller Spieler und keine vollständige taktische Rekonstruktion.


## 1. Setup

Die folgenden Zellen laden Pakete, setzen Pfade und definieren ein schlichtes, konsistentes Plot-Design. Die finalen Abbildungen werden in `GDV/figures/` gespeichert.


In [ ]:
from pathlib import Path
import ast
import math
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsbombpy import sb

warnings.filterwarnings("ignore")

cwd = Path.cwd()
if (cwd / "GDV").exists():
    REPO_ROOT = cwd
elif cwd.name == "notebooks" and cwd.parent.name == "GDV":
    REPO_ROOT = cwd.parents[1]
elif cwd.name == "GDV":
    REPO_ROOT = cwd.parent
else:
    REPO_ROOT = cwd

GDV_DIR = REPO_ROOT / "GDV"
DATA_DIR = GDV_DIR / "data"
FIGURE_DIR = GDV_DIR / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

COMPETITION_ID = 43
SEASON_ID = 106

GOAL_BUILDUPS_FILE = DATA_DIR / "goal_buildups.csv"
GOAL_PASSES_FILE = DATA_DIR / "goal_buildup_passes.csv"
FINAL_THIRD_ENTRIES_FILE = DATA_DIR / "final_third_entries.csv"
TEAM_ENTRY_SUMMARY_FILE = DATA_DIR / "team_entry_summary.csv"
FORMATION_FILE = DATA_DIR / "team_match_formations.csv"

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#2f2f2f"
plt.rcParams["axes.labelcolor"] = "#222222"
plt.rcParams["xtick.color"] = "#222222"
plt.rcParams["ytick.color"] = "#222222"
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titleweight"] = "bold"

COLOR_DIRECT = "#2F80ED"
COLOR_RULE = "#F2994A"
COLOR_BUILDUP = "#9B51E0"
COLOR_DARK = "#222222"
COLOR_LIGHT = "#E8EEF7"
COLOR_GOAL = "#EB5757"
COLOR_SHOT = "#27AE60"
COLOR_NONE = "#BDBDBD"

print("Repository:", REPO_ROOT)
print("GDV data:", DATA_DIR)
print("GDV figures:", FIGURE_DIR)

## 2. Hilfsfunktionen

StatsBomb speichert Positionen als Listen mit x- und y-Koordinaten. Die Hilfsfunktionen extrahieren diese Werte und berechnen Zeitpunkte in Sekunden.


In [ ]:
def parse_location(value):
    if isinstance(value, list) and len(value) >= 2:
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list) and len(parsed) >= 2:
                return parsed
        except Exception:
            return None
    return None


def get_x(value):
    loc = parse_location(value)
    if isinstance(loc, list):
        return loc[0]
    return np.nan


def get_y(value):
    loc = parse_location(value)
    if isinstance(loc, list):
        return loc[1]
    return np.nan


def seconds_total(row):
    minute = int(row["minute"]) if not pd.isna(row["minute"]) else 0
    second = int(row["second"]) if not pd.isna(row["second"]) else 0
    return minute * 60 + second


def pass_distance(x1, y1, x2, y2):
    if pd.isna(x1) or pd.isna(y1) or pd.isna(x2) or pd.isna(y2):
        return np.nan
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


def savefig(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Gespeichert:", path)

## 3. Datenaufbereitung: Torangriffe

Für jedes Tor wird die gleiche Ballbesitzphase des erzielenden Teams betrachtet. Innerhalb dieser Possession werden alle erfolgreichen Pässe vor dem Tor gezählt.

Das ist die Grundlage für die 4–5-Pass-Beobachtung: Wie viele Pässe braucht ein Angriff durchschnittlich, bis daraus ein Tor entsteht?


In [ ]:
def build_goal_sequences(force_rebuild=False):
    if GOAL_BUILDUPS_FILE.exists() and GOAL_PASSES_FILE.exists() and not force_rebuild:
        return pd.read_csv(GOAL_BUILDUPS_FILE), pd.read_csv(GOAL_PASSES_FILE)

    matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
    all_buildups = []
    all_passes = []

    for match_idx, match in matches.reset_index(drop=True).iterrows():
        match_id = int(match["match_id"])
        home_team = match["home_team"]
        away_team = match["away_team"]
        match_label = f"{home_team} vs {away_team}"
        print(f"{match_idx + 1}/{len(matches)} {match_label}")

        try:
            events = sb.events(match_id=match_id)
        except Exception as exc:
            print(f"Match {match_id} konnte nicht geladen werden: {exc}")
            continue

        needed_columns = [
            "id", "index", "period", "timestamp", "minute", "second", "team", "player", "type",
            "possession", "location", "pass_end_location", "pass_outcome", "shot_outcome", "play_pattern"
        ]
        for col in needed_columns:
            if col not in events.columns:
                events[col] = np.nan

        events = events.sort_values(["period", "index"]).copy()
        events["event_seconds"] = events.apply(seconds_total, axis=1)
        events["x"] = events["location"].apply(get_x)
        events["y"] = events["location"].apply(get_y)
        events["end_x"] = events["pass_end_location"].apply(get_x)
        events["end_y"] = events["pass_end_location"].apply(get_y)

        goals = events[(events["type"] == "Shot") & (events["shot_outcome"].astype(str).str.lower() == "goal")].copy()

        for goal_number, goal in goals.reset_index(drop=True).iterrows():
            team = goal["team"]
            possession = goal["possession"]
            goal_index = goal["index"]
            goal_seconds = goal["event_seconds"]
            build_up_id = f"{match_id}_{goal_number + 1}"

            same_possession = events[
                (events["possession"] == possession)
                & (events["team"] == team)
                & (events["index"] <= goal_index)
            ].copy()

            if same_possession.empty:
                continue

            successful_passes = same_possession[
                (same_possession["type"] == "Pass")
                & (same_possession["pass_outcome"].isna() | (same_possession["pass_outcome"].astype(str).str.lower() == "nan"))
            ].copy()

            if successful_passes.empty:
                first_event = same_possession.iloc[0]
                start_x = first_event["x"]
                start_y = first_event["y"]
                start_seconds = first_event["event_seconds"]
            else:
                first_pass = successful_passes.iloc[0]
                start_x = first_pass["x"]
                start_y = first_pass["y"]
                start_seconds = first_pass["event_seconds"]

            duration_seconds = max(0, goal_seconds - start_seconds)
            players_involved = same_possession[same_possession["player"].notna()]["player"].nunique()

            all_buildups.append({
                "build_up_id": build_up_id,
                "match_id": match_id,
                "match_date": match.get("match_date"),
                "match_label": match_label,
                "team": team,
                "goal_player": goal.get("player"),
                "goal_minute": goal.get("minute"),
                "goal_second": goal.get("second"),
                "period": goal.get("period"),
                "possession": possession,
                "play_pattern": goal.get("play_pattern"),
                "passes_before_goal": len(successful_passes),
                "duration_seconds": duration_seconds,
                "players_involved": players_involved,
                "start_x": start_x,
                "start_y": start_y,
                "goal_x": goal.get("x"),
                "goal_y": goal.get("y")
            })

            for pass_order, p in successful_passes.reset_index(drop=True).iterrows():
                distance = pass_distance(p.get("x"), p.get("y"), p.get("end_x"), p.get("end_y"))
                all_passes.append({
                    "build_up_id": build_up_id,
                    "match_id": match_id,
                    "match_label": match_label,
                    "team": team,
                    "goal_player": goal.get("player"),
                    "goal_minute": goal.get("minute"),
                    "pass_order": pass_order + 1,
                    "player": p.get("player"),
                    "minute": p.get("minute"),
                    "second": p.get("second"),
                    "x": p.get("x"),
                    "y": p.get("y"),
                    "end_x": p.get("end_x"),
                    "end_y": p.get("end_y"),
                    "pass_distance": distance,
                    "forward_gain": p.get("end_x") - p.get("x") if not pd.isna(p.get("end_x")) and not pd.isna(p.get("x")) else np.nan
                })

        time.sleep(0.05)

    buildups = pd.DataFrame(all_buildups)
    passes = pd.DataFrame(all_passes)
    buildups.to_csv(GOAL_BUILDUPS_FILE, index=False)
    passes.to_csv(GOAL_PASSES_FILE, index=False)
    return buildups, passes


buildups, passes = build_goal_sequences(force_rebuild=False)
print("Analysierte Tore:", len(buildups))
print("Pässe vor Toren:", len(passes))
display(buildups.head())

## 4. Angriffskategorien passend zur 4–5-Pass-Beobachtung

Die Kategorien sind bewusst auf die Forschungsfrage zugeschnitten. Der Bereich **4–5 Pässe** wird separat betrachtet, weil genau dieser Bereich aus dem Trainerumfeld als typische Anzahl Pässe vor Toren genannt wird. Die Analyse prüft also nicht eine feste Regel, sondern ob diese Beobachtung in den WM-Daten sichtbar ist.


In [ ]:
def attack_category(pass_count):
    if pass_count <= 3:
        return "0–3 Pässe: sehr direkt"
    if pass_count <= 5:
        return "4–5 Pässe: Beobachtung"
    if pass_count <= 8:
        return "6–8 Pässe: Aufbau"
    return "9+ Pässe: langer Aufbau"


def start_zone(x):
    if pd.isna(x):
        return "Unbekannt"
    if x < 40:
        return "Defensives Drittel"
    if x < 80:
        return "Mittleres Drittel"
    return "Offensives Drittel"


category_order = [
    "0–3 Pässe: sehr direkt",
    "4–5 Pässe: Beobachtung",
    "6–8 Pässe: Aufbau",
    "9+ Pässe: langer Aufbau"
]

buildups = buildups.copy()
passes = passes.copy()

buildups["attack_category"] = buildups["passes_before_goal"].apply(attack_category)
buildups["start_zone"] = buildups["start_x"].apply(start_zone)

if "attack_category" in passes.columns:
    passes = passes.drop(columns=["attack_category"])

passes = passes.merge(
    buildups[["build_up_id", "attack_category"]],
    on="build_up_id",
    how="left"
)

summary = buildups["attack_category"].value_counts().reindex(category_order).fillna(0).astype(int)
summary_percent = (summary / summary.sum() * 100).round(1)

print("Tore nach Angriffskategorie")
display(pd.DataFrame({"Tore": summary, "Anteil %": summary_percent}))
print("Mittelwert Pässe vor Tor:", round(buildups["passes_before_goal"].mean(), 2))
print("Median Pässe vor Tor:", round(buildups["passes_before_goal"].median(), 2))


## 5. Zusätzliche Analyse: schnelle Entries ins offensive Drittel

Für den praktischen Trainerbezug reicht es nicht, nur Tore zu zählen. Ein Team kann oft schnell ins offensive Drittel kommen, ohne daraus direkt ein Tor zu erzielen. Deshalb wird hier aus allen Eventdaten geprüft:

- Wann kommt der Ball erstmals in einer Possession ins offensive Drittel?
- Passiert das schnell?
- Entsteht danach ein Schuss oder Tor?
- Geschieht der Entry durch kurzen Pass, langen Ball oder Carry?

Definition: Offensives Drittel = `x >= 80` im StatsBomb-Koordinatensystem.


In [ ]:
def classify_entry_method(row):
    event_type = row.get("type")
    if event_type == "Carry":
        return "Carry / Dribbling"
    distance = row.get("entry_distance")
    gain = row.get("forward_gain")
    if pd.isna(distance):
        distance = 0
    if pd.isna(gain):
        gain = 0
    if distance >= 30 or gain >= 25:
        return "Langer Ball / langer Pass"
    return "Kurzer Pass / Kombination"


def extract_formation_value(value):
    if isinstance(value, dict):
        return value.get("formation")
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, dict):
                return parsed.get("formation")
        except Exception:
            return np.nan
    return np.nan


def build_final_third_entries(force_rebuild=False):
    if FINAL_THIRD_ENTRIES_FILE.exists() and TEAM_ENTRY_SUMMARY_FILE.exists() and FORMATION_FILE.exists() and not force_rebuild:
        entries = pd.read_csv(FINAL_THIRD_ENTRIES_FILE)
        team_summary = pd.read_csv(TEAM_ENTRY_SUMMARY_FILE)
        formations = pd.read_csv(FORMATION_FILE)
        return entries, team_summary, formations

    matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
    all_entries = []
    all_formations = []

    for match_idx, match in matches.reset_index(drop=True).iterrows():
        match_id = int(match["match_id"])
        match_label = f"{match['home_team']} vs {match['away_team']}"
        print(f"Entries {match_idx + 1}/{len(matches)} {match_label}")

        try:
            events = sb.events(match_id=match_id)
        except Exception as exc:
            print(f"Match {match_id} konnte nicht geladen werden: {exc}")
            continue

        for col in ["index", "period", "minute", "second", "team", "player", "type", "possession", "location", "pass_end_location", "carry_end_location", "pass_outcome", "shot_outcome", "tactics"]:
            if col not in events.columns:
                events[col] = np.nan

        events = events.sort_values(["period", "index"]).copy()
        events["event_seconds"] = events.apply(seconds_total, axis=1)
        events["x"] = events["location"].apply(get_x)
        events["y"] = events["location"].apply(get_y)
        events["pass_end_x"] = events["pass_end_location"].apply(get_x)
        events["pass_end_y"] = events["pass_end_location"].apply(get_y)
        events["carry_end_x"] = events["carry_end_location"].apply(get_x)
        events["carry_end_y"] = events["carry_end_location"].apply(get_y)
        events["formation"] = events["tactics"].apply(extract_formation_value)

        formation_events = events[(events["type"] == "Starting XI") & events["formation"].notna()].copy()
        for _, f in formation_events.iterrows():
            all_formations.append({
                "match_id": match_id,
                "match_label": match_label,
                "team": f["team"],
                "formation": str(int(f["formation"])) if not pd.isna(f["formation"]) else np.nan
            })

        for (period, possession, team), group in events.groupby(["period", "possession", "team"], dropna=True):
            group = group.sort_values("index").copy()
            if group.empty:
                continue

            possession_start = group["event_seconds"].min()
            entry_candidates = []

            pass_candidates = group[
                (group["type"] == "Pass")
                & (group["pass_outcome"].isna() | (group["pass_outcome"].astype(str).str.lower() == "nan"))
                & (group["x"] < 80)
                & (group["pass_end_x"] >= 80)
            ].copy()
            pass_candidates["end_x"] = pass_candidates["pass_end_x"]
            pass_candidates["end_y"] = pass_candidates["pass_end_y"]
            entry_candidates.append(pass_candidates)

            carry_candidates = group[
                (group["type"] == "Carry")
                & (group["x"] < 80)
                & (group["carry_end_x"] >= 80)
            ].copy()
            carry_candidates["end_x"] = carry_candidates["carry_end_x"]
            carry_candidates["end_y"] = carry_candidates["carry_end_y"]
            entry_candidates.append(carry_candidates)

            entry_candidates = pd.concat(entry_candidates, ignore_index=False) if entry_candidates else pd.DataFrame()
            if entry_candidates.empty:
                continue

            first_entry = entry_candidates.sort_values("index").iloc[0]
            before_entry = group[group["index"] <= first_entry["index"]]
            after_entry = group[group["index"] >= first_entry["index"]]

            pass_count_before_entry = before_entry[
                (before_entry["type"] == "Pass")
                & (before_entry["pass_outcome"].isna() | (before_entry["pass_outcome"].astype(str).str.lower() == "nan"))
            ].shape[0]

            seconds_to_entry = first_entry["event_seconds"] - possession_start
            players_before_entry = before_entry[before_entry["player"].notna()]["player"].nunique()
            shots_after_entry = after_entry[after_entry["type"] == "Shot"]
            has_shot = len(shots_after_entry) > 0
            has_goal = (shots_after_entry["shot_outcome"].astype(str).str.lower() == "goal").any() if has_shot else False
            entry_distance = pass_distance(first_entry["x"], first_entry["y"], first_entry["end_x"], first_entry["end_y"])
            forward_gain = first_entry["end_x"] - first_entry["x"] if not pd.isna(first_entry["end_x"]) and not pd.isna(first_entry["x"]) else np.nan
            is_fast_entry = (seconds_to_entry <= 15) or (pass_count_before_entry <= 5)

            all_entries.append({
                "match_id": match_id,
                "match_label": match_label,
                "period": period,
                "possession": possession,
                "team": team,
                "event_type": first_entry["type"],
                "minute": first_entry["minute"],
                "second": first_entry["second"],
                "x": first_entry["x"],
                "y": first_entry["y"],
                "end_x": first_entry["end_x"],
                "end_y": first_entry["end_y"],
                "seconds_to_entry": seconds_to_entry,
                "passes_before_entry": pass_count_before_entry,
                "players_before_entry": players_before_entry,
                "entry_distance": entry_distance,
                "forward_gain": forward_gain,
                "is_fast_entry": is_fast_entry,
                "has_shot_after_entry": has_shot,
                "has_goal_after_entry": has_goal
            })

        time.sleep(0.05)

    entries = pd.DataFrame(all_entries)
    formations = pd.DataFrame(all_formations).drop_duplicates()
    if not entries.empty:
        entries["entry_method"] = entries.apply(classify_entry_method, axis=1)

    team_summary = entries.groupby("team").agg(
        entries=("team", "count"),
        fast_entries=("is_fast_entry", "sum"),
        shots_after_entry=("has_shot_after_entry", "sum"),
        goals_after_entry=("has_goal_after_entry", "sum"),
        avg_seconds_to_entry=("seconds_to_entry", "mean"),
        avg_players_before_entry=("players_before_entry", "mean")
    ).reset_index() if not entries.empty else pd.DataFrame()

    if not team_summary.empty:
        team_summary["shot_rate_after_entry"] = team_summary["shots_after_entry"] / team_summary["entries"]
        team_summary["goal_rate_after_entry"] = team_summary["goals_after_entry"] / team_summary["entries"]
        team_summary["fast_entry_share"] = team_summary["fast_entries"] / team_summary["entries"]

    entries.to_csv(FINAL_THIRD_ENTRIES_FILE, index=False)
    team_summary.to_csv(TEAM_ENTRY_SUMMARY_FILE, index=False)
    formations.to_csv(FORMATION_FILE, index=False)
    return entries, team_summary, formations


entries, team_entry_summary, formations = build_final_third_entries(force_rebuild=False)
print("Entries ins offensive Drittel:", len(entries))
display(team_entry_summary.sort_values("fast_entries", ascending=False).head(10))

# Finale Visualisierungen

Die folgenden Abbildungen sind bewusst fokussiert. Es werden keine Heatmaps oder überladenen Passflusskarten als Hauptplots genutzt, weil sie die Trainerfrage weniger klar beantworten.


## Abbildung 1: 4–5-Pass-Beobachtung prüfen

Diese Grafik beantwortet die Kernfrage direkt. Der orange markierte Bereich zeigt Tore nach vier oder fünf Pässen. Dadurch sieht man sofort, ob die verbreitete Trainerbeobachtung in den WM-Daten stark vertreten ist.


In [ ]:
counts = buildups["passes_before_goal"].value_counts().sort_index()
max_passes = int(max(counts.index))
xs = list(range(0, max_passes + 1))
ys = [counts.get(x, 0) for x in xs]
colors = [COLOR_RULE if x in [4, 5] else COLOR_DIRECT for x in xs]

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(xs, ys, color=colors, edgecolor="white", linewidth=0.8)

for x, y_value in zip(xs, ys):
    if y_value > 0:
        ax.text(x, y_value + 0.4, str(int(y_value)), ha="center", va="bottom", fontsize=8)

rule_goals = int(buildups["passes_before_goal"].isin([4, 5]).sum())
total_goals = len(buildups)
rule_share = rule_goals / total_goals * 100
median_passes = buildups["passes_before_goal"].median()
mean_passes = buildups["passes_before_goal"].mean()

ax.axvline(
    median_passes,
    color=COLOR_DARK,
    linestyle="--",
    linewidth=1.5,
    label=f"Median: {median_passes:.0f}"
)

ax.set_title("Wie viele Pässe gehen einem WM-Tor voraus?")
ax.set_xlabel("Erfolgreiche Pässe vor dem Tor")
ax.set_ylabel("Anzahl Tore")
ax.set_xticks(xs)
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)

info_text = (
    f"4–5-Pass-Bereich: {rule_goals} von {total_goals} Toren ({rule_share:.1f}%)\n"
    f"Durchschnitt: {mean_passes:.2f} Pässe"
)

ax.text(
    0.98,
    0.95,
    info_text,
    transform=ax.transAxes,
    ha="right",
    va="top",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#cccccc")
)

savefig("gdv_final_01_4_5_pass_beobachtung.png")


## Abbildung 2: Angriffstypen rund um die 4–5-Pass-Beobachtung

Die Rohverteilung wird in vier verständliche Kategorien übersetzt. Dadurch sieht man, ob sehr direkte Angriffe, der 4–5-Pass-Bereich oder längerer Aufbau dominieren.


In [ ]:
cat_counts = buildups["attack_category"].value_counts().reindex(category_order).fillna(0)
cat_percent = cat_counts / cat_counts.sum() * 100
bar_colors = [COLOR_DIRECT, COLOR_RULE, "#56CCF2", COLOR_BUILDUP]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(cat_counts.index, cat_counts.values, color=bar_colors, edgecolor="white")

for bar, pct, count in zip(bars, cat_percent.values, cat_counts.values):
    label = f"{int(count)}\n{pct:.1f}%"
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        label,
        ha="center",
        va="bottom",
        fontsize=9
    )

ax.set_title("Welche Angriffstypen führen zu Toren?")
ax.set_xlabel("Angriffskategorie")
ax.set_ylabel("Anzahl Tore")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=18, ha="right")

savefig("gdv_final_02_attack_categories.png")


## Abbildung 3: Schnelle Entries ins offensive Drittel und Outcome

Diese Grafik nutzt nicht nur Tore, sondern alle erkannten Entries ins offensive Drittel. Sie zeigt, welche Teams den Ball häufig schnell in gefährliche Räume bringen und ob daraus anschliessend Schüsse oder Tore entstehen.

Wichtig: Das ist keine Kausalitätsanalyse. Die Grafik zeigt, welche Teams in den Eventdaten häufiger schnelle Progression und anschliessende Abschlüsse hatten.


In [ ]:
fast_entries = entries[entries["is_fast_entry"] == True].copy()

fast_team = fast_entries.groupby("team").agg(
    fast_entries=("team", "count"),
    shots_after_fast_entry=("has_shot_after_entry", "sum"),
    goals_after_fast_entry=("has_goal_after_entry", "sum"),
    avg_seconds_to_entry=("seconds_to_entry", "mean")
).reset_index()

fast_team["shot_rate_after_fast_entry"] = fast_team["shots_after_fast_entry"] / fast_team["fast_entries"]
fast_team["goal_rate_after_fast_entry"] = fast_team["goals_after_fast_entry"] / fast_team["fast_entries"]

fast_team = fast_team[fast_team["fast_entries"] >= 5].sort_values("fast_entries", ascending=True).tail(12)

fig, ax = plt.subplots(figsize=(11, 7))
y_pos = np.arange(len(fast_team))

bars = ax.barh(
    y_pos,
    fast_team["fast_entries"],
    color=COLOR_DIRECT,
    edgecolor="white"
)

for i, (_, row) in enumerate(fast_team.iterrows()):
    text = (
        f"{int(row['fast_entries'])} Entries | "
        f"{row['shot_rate_after_fast_entry'] * 100:.0f}% Schussrate | "
        f"{int(row['goals_after_fast_entry'])} Tore"
    )
    ax.text(row["fast_entries"] + 0.4, i, text, va="center", fontsize=8)

ax.set_yticks(y_pos)
ax.set_yticklabels(fast_team["team"])
ax.set_title("Welche Teams kommen häufig schnell ins offensive Drittel?")
ax.set_xlabel("Schnelle Entries ins offensive Drittel")
ax.set_ylabel("Team")
ax.grid(axis="x", alpha=0.25)

ax.text(
    0.02,
    0.96,
    "Schnell = innerhalb von 15 Sekunden oder mit höchstens 5 Pässen bis zum Entry",
    transform=ax.transAxes,
    va="top",
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="#cccccc")
)

savefig("gdv_final_03_fast_entries_outcomes.png")


## Abbildung 4: Entry-Methode und Ergebnis

Hier wird untersucht, wie der Ball ins offensive Drittel kommt: per langer Ball, kurzer Kombination oder Carry. Dadurch wird die Frage nach direktem Spielaufbau und langen Bällen konkreter.


In [ ]:
entry_data = entries.copy()
entry_data["entry_outcome"] = np.where(
    entry_data["has_goal_after_entry"], "Tor nach Entry",
    np.where(entry_data["has_shot_after_entry"], "Schuss ohne Tor", "Kein Schuss")
)

method_order = ["Langer Ball / langer Pass", "Kurzer Pass / Kombination", "Carry / Dribbling"]
outcome_order = ["Kein Schuss", "Schuss ohne Tor", "Tor nach Entry"]
method_counts = pd.crosstab(entry_data["entry_method"], entry_data["entry_outcome"]).reindex(method_order).fillna(0)
for col in outcome_order:
    if col not in method_counts.columns:
        method_counts[col] = 0
method_counts = method_counts[outcome_order]
method_share = method_counts.div(method_counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 6))
left = np.zeros(len(method_share))
colors = [COLOR_NONE, COLOR_SHOT, COLOR_GOAL]
for outcome, color in zip(outcome_order, colors):
    values = method_share[outcome].values
    ax.barh(method_share.index, values, left=left, label=outcome, color=color, edgecolor="white")
    for i, value in enumerate(values):
        if value >= 7:
            ax.text(left[i] + value / 2, i, f"{value:.0f}%", ha="center", va="center", color="white" if outcome != "Kein Schuss" else "#222222", fontsize=8)
    left += values

ax.set_title("Wie effizient sind unterschiedliche Wege ins offensive Drittel?")
ax.set_xlabel("Anteil der Entries (%)")
ax.set_ylabel("Entry-Methode")
ax.legend(frameon=False, loc="lower right")
ax.set_xlim(0, 100)
ax.grid(axis="x", alpha=0.2)

savefig("gdv_final_04_entry_method_efficiency.png")

## Abbildung 5: Startzone nach Angriffstyp

Diese Grafik ersetzt die früheren Heatmaps. Statt schwer interpretierbarer Flächen zeigt sie direkt, ob direkte Tore eher aus tiefen, mittleren oder hohen Startzonen entstehen.


In [ ]:
start_table = pd.crosstab(buildups["attack_category"], buildups["start_zone"]).reindex(category_order).fillna(0)
zone_order = ["Defensives Drittel", "Mittleres Drittel", "Offensives Drittel", "Unbekannt"]
for col in zone_order:
    if col not in start_table.columns:
        start_table[col] = 0
start_table = start_table[zone_order]
start_share = start_table.div(start_table.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 6))
left = np.zeros(len(start_share))
zone_colors = ["#2D9CDB", "#F2C94C", "#EB5757", "#BDBDBD"]
for zone, color in zip(zone_order, zone_colors):
    values = start_share[zone].values
    ax.barh(start_share.index, values, left=left, label=zone, color=color, edgecolor="white")
    left += values

ax.set_title("Wo beginnen die Torangriffe je Angriffstyp?")
ax.set_xlabel("Anteil der Tore (%)")
ax.set_ylabel("Angriffstyp")
ax.legend(frameon=False, loc="lower right")
ax.set_xlim(0, 100)
ax.grid(axis="x", alpha=0.2)

savefig("gdv_final_05_start_zone_by_attack_type.png")

## Abbildung 6: Teamranking – wer trifft am direktesten?

Für den Trainer-Use-Case ist ein Ranking gut erklärbar: Teams links bzw. oben kommen im Schnitt mit weniger Pässen zum Tor. Der orange Bereich markiert die 4–5-Pass-Beobachtung.


In [ ]:
team_directness = buildups.groupby("team").agg(
    goals=("build_up_id", "count"),
    avg_passes=("passes_before_goal", "mean"),
    median_passes=("passes_before_goal", "median"),
    avg_duration=("duration_seconds", "mean")
).reset_index()

team_directness = team_directness[team_directness["goals"] >= 2].sort_values("avg_passes", ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
y = np.arange(len(team_directness))

ax.hlines(y, xmin=0, xmax=team_directness["avg_passes"], color=COLOR_LIGHT, linewidth=5)
ax.scatter(team_directness["avg_passes"], y, color=COLOR_DIRECT, s=70, zorder=3)

ax.axvspan(4, 5, color=COLOR_RULE, alpha=0.18, label="4–5-Pass-Bereich")

for i, (_, row) in enumerate(team_directness.iterrows()):
    ax.text(
        row["avg_passes"] + 0.08,
        i,
        f"{row['avg_passes']:.1f}  ({int(row['goals'])} Tore)",
        va="center",
        fontsize=8
    )

ax.set_yticks(y)
ax.set_yticklabels(team_directness["team"])
ax.invert_yaxis()
ax.set_title("Welche Teams erzielen ihre Tore am direktesten?")
ax.set_xlabel("Durchschnittliche Pässe vor dem Tor")
ax.set_ylabel("Team")
ax.grid(axis="x", alpha=0.25)
ax.legend(frameon=False, loc="lower right")

savefig("gdv_final_06_team_directness_ranking.png")


## Abbildung 7: Case Study im 4–5-Pass-Bereich

Die Fallstudie zeigt ein konkretes Tor mit vier oder fünf Pässen. Sie macht sichtbar, was hinter der abstrakten Zahl steckt.


In [ ]:
def draw_pitch(ax):
    ax.plot([0, 120], [0, 0], color="#333333", linewidth=1)
    ax.plot([0, 120], [80, 80], color="#333333", linewidth=1)
    ax.plot([0, 0], [0, 80], color="#333333", linewidth=1)
    ax.plot([120, 120], [0, 80], color="#333333", linewidth=1)
    ax.plot([60, 60], [0, 80], color="#333333", linewidth=1)
    ax.add_patch(plt.Circle((60, 40), 10, fill=False, color="#333333", linewidth=1))
    ax.plot([0, 18], [18, 18], color="#333333", linewidth=1)
    ax.plot([18, 18], [18, 62], color="#333333", linewidth=1)
    ax.plot([18, 0], [62, 62], color="#333333", linewidth=1)
    ax.plot([120, 102], [18, 18], color="#333333", linewidth=1)
    ax.plot([102, 102], [18, 62], color="#333333", linewidth=1)
    ax.plot([102, 120], [62, 62], color="#333333", linewidth=1)
    ax.set_xlim(0, 120)
    ax.set_ylim(80, 0)
    ax.set_aspect("equal")
    ax.axis("off")

four_five_pass_goals = buildups[buildups["passes_before_goal"].isin([4, 5])].copy()
four_five_pass_goals = four_five_pass_goals.sort_values(["passes_before_goal", "duration_seconds"], ascending=[False, False])
case_goal = four_five_pass_goals.iloc[0]
case_id = case_goal["build_up_id"]
case_passes = passes[passes["build_up_id"] == case_id].sort_values("pass_order").copy()

fig, ax = plt.subplots(figsize=(11, 7))
draw_pitch(ax)

for _, p in case_passes.iterrows():
    ax.annotate(
        "",
        xy=(p["end_x"], p["end_y"]),
        xytext=(p["x"], p["y"]),
        arrowprops=dict(arrowstyle="->", color=COLOR_DIRECT, lw=2, alpha=0.85)
    )
    ax.scatter(p["x"], p["y"], s=45, color="white", edgecolor=COLOR_DIRECT, linewidth=1.5, zorder=3)
    ax.text(p["x"], p["y"], str(int(p["pass_order"])), ha="center", va="center", fontsize=8, color=COLOR_DARK, zorder=4)

if not case_passes.empty:
    first = case_passes.iloc[0]
    ax.scatter(first["x"], first["y"], s=130, color=COLOR_RULE, edgecolor="black", linewidth=1.2, label="Start")

ax.scatter(case_goal["goal_x"], case_goal["goal_y"], marker="*", s=420, color=COLOR_GOAL, edgecolor="black", linewidth=1.2, label="Tor")

ax.set_title(f"Case Study: {case_goal['team']} – {case_goal['passes_before_goal']} Pässe vor dem Tor")
ax.text(2, 84, f"Spiel: {case_goal['match_label']} | Torschütze: {case_goal['goal_player']} | Minute: {int(case_goal['goal_minute'])}", fontsize=10)
ax.legend(loc="lower center", ncol=2, frameon=False)

savefig("gdv_final_07_4_5_pass_case_study.png")

display(case_passes[["pass_order", "player", "x", "y", "end_x", "end_y", "forward_gain"]])


# Explorative Zusatzanalyse: Formation

Formation ist für die Fragestellung interessant, aber methodisch vorsichtig zu interpretieren. In Eventdaten wird die Formation über Starting-XI-Events abgeleitet. Das zeigt die Grundordnung, aber nicht automatisch die tatsächliche Staffelung während einer Aktion.

Darum wird dieser Plot nicht als Hauptbeweis verwendet, sondern als zusätzlicher Kontext.


In [ ]:
if not formations.empty:
    goal_formations = buildups.merge(formations[["match_id", "team", "formation"]], on=["match_id", "team"], how="left")
    formation_counts = pd.crosstab(goal_formations["formation"], goal_formations["attack_category"])
    if not formation_counts.empty:
        for col in category_order:
            if col not in formation_counts.columns:
                formation_counts[col] = 0
        formation_counts = formation_counts[category_order]
        formation_counts = formation_counts[formation_counts.sum(axis=1) >= 3]
        formation_share = formation_counts.div(formation_counts.sum(axis=1), axis=0) * 100

        fig, ax = plt.subplots(figsize=(10, 6))
        left = np.zeros(len(formation_share))
        for col, color in zip(category_order, [COLOR_DIRECT, COLOR_RULE, "#56CCF2", COLOR_BUILDUP]):
            values = formation_share[col].values
            ax.barh(formation_share.index.astype(str), values, left=left, label=col, color=color, edgecolor="white")
            left += values
        ax.set_title("Zusatzanalyse: Angriffstypen nach Grundformation")
        ax.set_xlabel("Anteil der Tore (%)")
        ax.set_ylabel("Formation")
        ax.legend(frameon=False, fontsize=8, loc="lower right")
        ax.set_xlim(0, 100)
        ax.grid(axis="x", alpha=0.2)
        savefig("gdv_final_08_formation_context.png")
    else:
        print("Keine ausreichenden Formationsdaten für den Plot.")
else:
    print("Keine Formationsdaten gefunden.")

# Zusammenfassung

Die Analyse prüft die 4–5-Pass-Beobachtung nicht nur über eine einzelne Zahl, sondern über mehrere Blickwinkel:

- Die Passverteilung zeigt, wie häufig vier bis fünf Pässe vor einem Tor vorkommen.
- Die Angriffskategorien zeigen, ob sehr direkte Angriffe oder längerer Aufbau dominieren.
- Die Entry-Analyse zeigt, welche Teams schnell ins offensive Drittel kommen und ob daraus Abschlüsse entstehen.
- Die Entry-Methode zeigt, ob lange Bälle, kurze Kombinationen oder Carries effizienter wirken.
- Die Startzonen zeigen, wo Torangriffe typischerweise beginnen.
- Das Teamranking macht sichtbar, welche Teams besonders direkt zum Tor kommen.
- Die Case Study übersetzt die Zahl in eine konkrete Passsequenz.

Wichtig: Die Analyse liefert praktische Hinweise für Training und Spielidee, aber keine vollständige taktische Wahrheit. Dafür wären zusätzlich Trackingdaten und Kontextinformationen nötig.


In [ ]:
final_files = [
    "gdv_final_01_4_5_pass_beobachtung.png",
    "gdv_final_02_attack_categories.png",
    "gdv_final_03_fast_entries_outcomes.png",
    "gdv_final_04_entry_method_efficiency.png",
    "gdv_final_05_start_zone_by_attack_type.png",
    "gdv_final_06_team_directness_ranking.png",
    "gdv_final_07_4_5_pass_case_study.png"
]

print("Finale GDV-Figuren")
print("------------------")
for file in final_files:
    print(FIGURE_DIR / file)

rule_goals = buildups["passes_before_goal"].isin([4, 5]).sum()
print()
print(f"Analysierte Tore: {len(buildups)}")
print(f"Tore mit 4–5 Pässen: {rule_goals} ({rule_goals / len(buildups) * 100:.1f}%)")
print(f"Median Pässe vor Tor: {buildups['passes_before_goal'].median():.0f}")
print(f"Durchschnitt Pässe vor Tor: {buildups['passes_before_goal'].mean():.2f}")
